In [1]:
import numpy as np
import itertools
from tqdm import tqdm
import rasterio as rio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.warp import reproject
from matplotlib import pyplot as plt
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"

In [4]:
wsf3d_material_stock_ = rio.open(DATA / 'WSF3D_V02_MaterialStock_minerals.tif')

In [5]:
wsf3d_material_stock_.shape

(179541, 462857)

In [6]:
H, W = wsf3d_material_stock_.shape
k = 10000
the_negative = -32767
r = 10
q = int(k // r)
I = H // k + 1
J = W // k + 1 

new_array = np.zeros((I * q, J * q), dtype=np.float64)
# new_array = np.zeros((I * q, J * q), dtype=np.int64)
# the_sum = 0 

for idx in tqdm(itertools.product(range(I), range(J))):
    i, j = idx
    m = k if i < (I-1) else H - i * k
    n = k if j < (J-1) else W - j * k
    a_window = ((i*k, i*k+m), (j*k, j*k+n))
    a_slice = wsf3d_material_stock_.read(1, window=a_window)

    # a_sum = np.sum(np.where(a_slice >=0, a_slice, 0) / 1e3).astype(np.float64)  # unit: kg --> t
    # the_sum += a_sum    
    aa = np.vstack([a_slice, np.zeros((k - a_slice.shape[0], a_slice.shape[1]))])
    bb = np.hstack([aa, np.zeros((aa.shape[0], k - a_slice.shape[1]))])
    cc = np.where(bb > 0, bb, 0)
    dd = cc.reshape(k // r, r, k // r, r).sum(axis=(1, 3)) / 1e3     # unit: kg --> t

    new_array[i*q:(i+1)*q, j*q:(j+1)*q] = dd 
    # print(a_window)

846it [12:25,  1.13it/s]


In [8]:
new_array[new_array > 0].sum() / 1e9 # unit: Gt

np.float64(401.41428413466804)

In [10]:
global_building_minerals = np.zeros((21600, 43200))
reproject(
        source=new_array[:, :46286],
        destination=global_building_minerals,
        src_transform=Affine(0.00777777777778, 0.0, -180.0007222, 0.0, -0.00777777777778, 83.6590555),
        src_crs="EPSG:4326",
        dst_transform=Affine(1/120, 0, -180, 0, -1/120, 90),
        dst_crs="EPSG:4326",
        resampling=Resampling.sum
    )

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(21600, 43200)),
 Affine(0.008333333333333333, 0.0, -180.0,
        0.0, -0.008333333333333333, 90.0))

In [12]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [13]:
global_building_minerals_1km = np.zeros(country_raster_1km_.shape, dtype=np.float64)
reproject(
        source=global_building_minerals,
        destination=global_building_minerals_1km,
        src_transform=Affine(1/120, 0, -180, 0, -1/120, 90),
        src_crs="EPSG:4326",
        dst_transform=country_raster_1km_.transform,
        dst_crs=country_raster_1km_.crs, 
        resampling=Resampling.sum
    )

print(global_building_minerals_1km[global_building_minerals_1km > 0].sum() / 1e9) # unit: Gt 

401.4170411640266


In [18]:
with rio.open(
    FEATURE_DIR / 'building-mineral-stock-1km.tif',
    "w",
    driver="GTiff",
    height=country_raster_1km_.shape[0],
    width=country_raster_1km_.shape[1],
    count=1,
    dtype=np.float32,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(global_building_minerals_1km.astype(np.float32), 1)